In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Embedding, Dense, Dropout, LayerNormalization, MultiHeadAttention
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow:", tf.__version__)


TensorFlow: 2.21.0


In [2]:
dataset = pd.read_csv("transformer_qa_49_rows.csv")

questions = dataset["question"].astype(str).tolist()
contexts = dataset["context"].astype(str).tolist()
answers = dataset["answer"].astype(str).tolist()

print("QA examples:", len(dataset))

for i in range(min(5, len(dataset))):
    print("Question:", questions[i])
    print("Context:", contexts[i])
    print("Answer:", answers[i])
    print()


QA examples: 49
Question: What is artificial intelligence?
Context: Artificial intelligence (AI) is the field of computer science focused on creating systems that can perform tasks that normally require human intelligence.
Answer: is the field of computer science focused on creating systems that can perform tasks

Question: What is machine learning?
Context: Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions or decisions.
Answer: Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data

Question: What is deep learning?
Context: Deep learning is a type of machine learning that uses neural networks with multiple layers to learn complex patterns from data.
Answer: uses neural networks with multiple layers to learn complex patterns

Question: What is NLP?
Context: Natural language processing, or NLP, is a field of AI that enables computers to understand, process, and

In [3]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<cls>", "<sep>"]

tokenizer = Tokenizer(
    filters="",
    lower=True,
    oov_token="<unk>",
)
tokenizer.fit_on_texts(questions + contexts)

word_index = tokenizer.word_index

for token in ["<pad>", "<cls>", "<sep>"]:
    if token not in word_index:
        word_index[token] = len(word_index) + 1

index_to_word = {idx: word for word, idx in word_index.items()}

PAD_ID = 0
UNK_ID = word_index["<unk>"]
CLS_ID = word_index["<cls>"]
SEP_ID = word_index["<sep>"]
vocab_size = max(word_index.values()) + 1

print("Vocabulary size:", vocab_size)
print("CLS:", CLS_ID, "SEP:", SEP_ID, "PAD:", PAD_ID)


Vocabulary size: 421
CLS: 419 SEP: 420 PAD: 0


In [4]:
def tokenize_text(text):
    return tokenizer.texts_to_sequences([text])[0]

def build_input(question, context):
    question_ids = tokenize_text(question)
    context_ids = tokenize_text(context)

    input_ids = [CLS_ID] + question_ids + [SEP_ID] + context_ids + [SEP_ID]

    context_start = 1 + len(question_ids) + 1
    context_end = context_start + len(context_ids) - 1

    return input_ids, context_start, context_end

sample = build_input(questions[0], contexts[0])
print("Sample input IDs:", sample[0])
print("Context starts at:", sample[1])
print("Context ends at:", sample[2])


Sample input IDs: [419, 4, 3, 50, 143, 420, 50, 88, 181, 3, 5, 89, 8, 182, 183, 184, 16, 90, 91, 12, 22, 185, 186, 12, 187, 188, 92, 189, 420]
Context starts at: 6
Context ends at: 27


In [5]:
def find_subsequence(sequence, target):
    if not target or len(target) > len(sequence):
        return -1

    for i in range(len(sequence) - len(target) + 1):
        if sequence[i:i + len(target)] == target:
            return i

    return -1


input_id_lists = []
start_positions = []
end_positions = []

for question, context, answer in zip(questions, contexts, answers):

    input_ids, context_start, context_end = build_input(question, context)

    context_ids = tokenize_text(context)
    answer_ids = tokenize_text(answer)

    relative_start = find_subsequence(context_ids, answer_ids)

    if relative_start == -1:
        raise ValueError(f"Answer not found in context: {answer}")

    start = context_start + relative_start
    end = start + len(answer_ids) - 1

    input_id_lists.append(input_ids)
    start_positions.append(start)
    end_positions.append(end)

print("Answer spans created successfully.")
print("First start:", start_positions[0])
print("First end:", end_positions[0])


Answer spans created successfully.
First start: 9
First end: 22


In [6]:
max_seq_length = max(len(x) for x in input_id_lists)

input_sequences = pad_sequences(
    input_id_lists,
    maxlen=max_seq_length,
    padding="post",
    truncating="post",
)

start_positions = np.array(start_positions, dtype="int32")
end_positions = np.array(end_positions, dtype="int32")

print("Max sequence length:", max_seq_length)
print("Input shape:", input_sequences.shape)


Max sequence length: 30
Input shape: (49, 30)


In [7]:
class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(self, vocab_size, sequence_length, d_model, **kwargs):
        super().__init__(**kwargs)
        self.token_embedding = Embedding(vocab_size, d_model, mask_zero=True)
        self.position_embedding = Embedding(sequence_length, d_model)

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(0, length)
        return (
            self.token_embedding(inputs)
            + self.position_embedding(positions)
        )


In [8]:
class TransformerEncoderBlock(tf.keras.layers.Layer):

    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout,
        )

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(d_model),
        ])

        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)

        self.dropout1 = Dropout(dropout)
        self.dropout2 = Dropout(dropout)

    def call(self, x, training=False, attention_mask=None):

        attention_output = self.attention(
            query=x,
            key=x,
            value=x,
            attention_mask=attention_mask,
            training=training,
        )

        x = self.norm1(
            x + self.dropout1(attention_output, training=training)
        )

        ffn_output = self.ffn(x, training=training)

        x = self.norm2(
            x + self.dropout2(ffn_output, training=training)
        )

        return x


In [9]:
class TransformerQA(tf.keras.Model):

    def __init__(
        self,
        vocab_size,
        max_sequence_length,
        d_model=64,
        num_heads=4,
        ff_dim=128,
        dropout=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.embedding = PositionalEmbedding(
            vocab_size,
            max_sequence_length,
            d_model,
        )

        self.encoder = TransformerEncoderBlock(
            d_model,
            num_heads,
            ff_dim,
            dropout,
        )

        self.start_head = Dense(1, name="start_head")
        self.end_head = Dense(1, name="end_head")

    def call(self, inputs, training=False):

        x = self.embedding(inputs)

        padding = tf.cast(inputs != 0, tf.int32)

        attention_mask = (
            padding[:, :, None] * padding[:, None, :]
        )

        x = self.encoder(
            x,
            training=training,
            attention_mask=attention_mask,
        )

        start_logits = tf.squeeze(
            self.start_head(x), axis=-1
        )
        end_logits = tf.squeeze(
            self.end_head(x), axis=-1
        )

        return start_logits, end_logits


In [10]:
D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

qa_model = TransformerQA(
    vocab_size=vocab_size,
    max_sequence_length=max_seq_length,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
)

dummy_start, dummy_end = qa_model(
    tf.constant(input_sequences[:1], dtype=tf.int32)
)

print("Start logits:", dummy_start.shape)
print("End logits:", dummy_end.shape)
qa_model.summary()


Start logits: (1, 30)
End logits: (1, 30)


Model: "transformer_qa"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ positional_embedding            │ ?                      │        28,864 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block       │ ?                      │        33,472 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ start_head (Dense)              │ (1, 30, 1)             │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ end_head (Dense)                │ (1, 30, 1)             │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,466 (244.01 KB)

 Trainable params: 62,466 (244.01 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

qa_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss_fn,
)


In [13]:
EPOCHS = 100
BATCH_SIZE = 8

history = qa_model.fit(
    input_sequences,
    (start_positions, end_positions),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    shuffle=True,
    verbose=1,
)


Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 6.4071 - sparse_categorical_crossentropy_loss: 3.3098
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.9875 - sparse_categorical_crossentropy_loss: 2.1841
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.9547 - sparse_categorical_crossentropy_loss: 2.0398
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.1018 - sparse_categorical_crossentropy_loss: 1.1469
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.5428 - sparse_categorical_crossentropy_loss: 0.7914
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.1202 - sparse_categorical_crossentropy_loss: 0.4734
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7807 - sparse_categorical_crossentropy_loss: 0.3284
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5893 - sparse_categorical_crossentropy_loss: 0.2585
Epoch 9/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4573 - sparse_categorical_crossentropy_loss: 0.1811
E

In [14]:
def decode_tokens(token_ids):
    words = []

    for token_id in token_ids:
        token_id = int(token_id)

        if token_id == 0:
            continue

        word = index_to_word.get(token_id, "<unk>")

        if word in ["<pad>", "<cls>", "<sep>"]:
            continue

        words.append(word)

    return " ".join(words)


def answer_question(question, context):

    input_ids, context_start, context_end = build_input(
        question,
        context,
    )

    padded = pad_sequences(
        [input_ids],
        maxlen=max_seq_length,
        padding="post",
        truncating="post",
    )

    start_logits, end_logits = qa_model(
        tf.constant(padded, dtype=tf.int32),
        training=False,
    )

    start_scores = start_logits[0].numpy()
    end_scores = end_logits[0].numpy()

    # Only answer from the context.
    start_scores[:context_start] = -1e9
    start_scores[context_end + 1:] = -1e9

    end_scores[:context_start] = -1e9
    end_scores[context_end + 1:] = -1e9

    best_start = int(np.argmax(start_scores))

    # End must be at or after start.
    end_scores[:best_start] = -1e9

    best_end = int(np.argmax(end_scores))

    answer_ids = padded[0][best_start:best_end + 1]

    return decode_tokens(answer_ids)


In [15]:
for i in range(min(10, len(dataset))):

    predicted = answer_question(
        questions[i],
        contexts[i],
    )

    print("Question :", questions[i])
    print("Expected :", answers[i])
    print("Predicted:", predicted)
    print("-" * 80)


Question : What is artificial intelligence?
Expected : is the field of computer science focused on creating systems that can perform tasks
Predicted: is the field of computer science focused on creating systems that can perform tasks
--------------------------------------------------------------------------------
Question : What is machine learning?
Expected : Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data
Predicted: machine learning is a branch of artificial intelligence that allows computers to learn patterns from data
--------------------------------------------------------------------------------
Question : What is deep learning?
Expected : uses neural networks with multiple layers to learn complex patterns
Predicted: uses neural networks with multiple layers to learn complex patterns
--------------------------------------------------------------------------------
Question : What is NLP?
Expected : enables computers to unde

## 14. Ask a New QA Question

In [16]:
new_context = (
    "Machine learning is a branch of artificial intelligence "
    "that allows computers to learn patterns from data and "
    "make predictions or decisions."
)

new_question = "machine is wat?"

print("Question:", new_question)
print("Context:", new_context)
print("Answer:", answer_question(new_question, new_context))


Question: machine is wat?
Context: Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions or decisions.
Answer: machine learning is a branch of artificial intelligence that allows computers to learn patterns
